# Case 3 · Does CareBridge reduce readmissions? (Part 2)
**BUS-G 492 · Causal decisions with AI**

In [ ]:
# --- Setup: run this cell first -----------------------------------------------
import os
DATA_URL = "https://raw.githubusercontent.com/boyoung-seo/492-coding/main/cases/case3_matching_care/data/"
for candidate in ["data/", "cases/case3_matching_care/data/", "../../../cases/case3_matching_care/data/", "../../../../cases/case3_matching_care/data/"]:
    if os.path.exists(candidate + "patients_2025.csv"):
        DATA = candidate
        break
else:
    DATA = DATA_URL
print("Reading data from:", DATA)

## Stage 5 · Something you didn't know

> **From:** Tom Adeyemi, RN, Discharge Nurse Manager
> **Re:** how we choose CareBridge patients
>
> We enroll the patients who scare us: **high severity**, and especially anyone who is **both very severe (60+) and has 3 or more chronic conditions**. Those are the ones who bounce back. Everything we look at is in your data.
>
> One exception: **St. Mary's (H07)** decided to enroll **every** discharged patient starting January 2025. Here are monthly readmission rates for all 20 hospitals since 2022: `hospital_monthly.csv`.

### Load the data

This is a new notebook, so nothing from Part 1 is loaded yet. Run this cell to load the Part 1 data again, then the next cell for the new files.

In [ ]:
import statsmodels.formula.api as smf
import pandas as pd, numpy as np
df = pd.read_csv(DATA + "patients_2025.csv")
print(df.shape)
df.head()

In [ ]:
hm = pd.read_csv(DATA + "hospital_monthly.csv")
hm.head()

## Your tasks

Use AI again, but this time **tell it the whole story**: how the data were generated, and what the new information is. Each step ends with a table: fill it in (double-click the cell to type) before moving on.

### Step 1 · Overlap: who has a comparison?

Leave out St. Mary's (`hospital == "H07"`), where everyone was enrolled. Plot the distribution of `severity_score` for enrolled and non-enrolled patients (two histograms on one chart), then fill in:

| Severity score | Share of patients enrolled |
|---|---|
| below 30 | |
| 80 and above | |

Where do both groups exist? Where are comparisons hard?

In [ ]:
x = df[df.hospital != "H07"].copy()
# Your code here (hint: x[x.enrolled_carebridge == 1].severity_score.hist(alpha=0.5) and the same for 0)


### Step 2 · Compare like with like (stratification) → ETT

Put patients into cells of similar patients, compare readmission rates of enrolled vs non-enrolled **inside each cell**, then average the differences **weighted by the number of enrolled patients** in each cell. That weighted average is the **effect on the treated (ETT)**. Drop cells that have no enrolled or no non-enrolled patients.

| Cells defined by | ETT (percentage points) | Share of enrollees with a comparable patient |
|---|---|---|
| severity (5-point bands) | | |
| severity (5-point bands) × number of chronic conditions | | |

Why does adding chronic conditions change the answer so much?

In [ ]:
x["sev5"] = x.severity_score // 5

def stratified(cols):
    cells = (x.groupby(cols + ["enrolled_carebridge"]).readmitted_30d
               .agg(["mean", "size"]).unstack())
    covered = cells[("size", 1)].sum()          # enrollees before dropping cells
    cells = cells.dropna()                      # keep cells with both groups
    diff = cells[("mean", 1)] - cells[("mean", 0)]
    w = cells[("size", 1)]                      # weight by enrollees
    return (diff * w).sum() / w.sum(), w.sum() / covered

# Your code here: call stratified(["sev5"]) and stratified(["sev5", "n_chronic_conditions"])


### Step 3 · Synthetic St. Mary's

St. Mary's (H07) enrolled **every** patient from January 2025. The code below builds a **synthetic St. Mary's**: a weighted mix of the other hospitals (weights ≥ 0, summing to 1) that tracks St. Mary's monthly readmission rate in 2022–2024. Run it, plot both lines, and fill in:

| 2025 | Readmission rate |
|---|---|
| St. Mary's (actual) | |
| Synthetic St. Mary's (what would have happened) | |
| Difference = ETT for St. Mary's patients | |

Which hospitals get the biggest weights? Why is this ETT smaller than the one in Step 2?

In [ ]:
from scipy.optimize import minimize
import numpy as np
wide = hm.pivot(index="month", columns="hospital", values="readmission_rate")
pre, post = wide.index < "2025-01", wide.index >= "2025-01"
donors = [h for h in wide.columns if h != "H07"]
Y, X = wide.loc[pre, "H07"].values, wide.loc[pre, donors].values
res = minimize(lambda w: ((Y - X @ w) ** 2).sum(), np.ones(len(donors)) / len(donors),
               bounds=[(0, 1)] * len(donors), constraints={"type": "eq", "fun": lambda w: w.sum() - 1})
weights = pd.Series(res.x, donors).round(3)
synth = wide[donors] @ res.x
print(weights[weights > 0.02].sort_values(ascending=False))
# Your code here: plot wide["H07"] and synth over time; compare their 2025 averages


### Step 4 · Into dollars → choose S

Repeat Step 2's stratified ETT **within each severity band**, then value it: each readmission avoided saves **$15,000**, and each enrolled patient costs **$600**.

| Severity | ETT (pts) | Value per enrolled patient = −ETT × $15,000 − $600 |
|---|---|---|
| below 40 | | |
| 40–49 | | |
| 50–59 | | |
| 60–69 | | |
| 70–79 | | |
| 80 and above | | |

Enroll the bands where the value is positive → choose S. Predicted profit = Σ (value per patient × patients in that band next year); next year has 20,000 discharges with about the same severity mix as this year.

In [ ]:
# Your code here (hint: loop over bands; inside each band, use the same cell logic as stratified(["sev5", "n_chronic_conditions"]))


### Decide again

In [ ]:
TEAM = ""                  # e.g. "Team 3: Ana & Ben"  (use the same name every time)
MIN_SEVERITY = None        # enroll patients with severity_score >= this; 101 = scrap the program  (e.g. 50)
PREDICTED_PROFIT = None    # in dollars, as a number, e.g. 150000
JUSTIFICATION = """
Three sentences for the decision maker: what you found, what you recommend, and why.
"""

# Stops with a message if anything is still blank, so you don't submit an empty field.
blank = [name for name, value in [("TEAM", TEAM), ("MIN_SEVERITY", MIN_SEVERITY), ("PREDICTED_PROFIT", PREDICTED_PROFIT)] if value in (None, "")]
if blank:
    raise ValueError("Fill these in before submitting: " + ", ".join(blank))
PREDICTED_PROFIT = float(PREDICTED_PROFIT)   # must be a number
print("Enter exactly these values in the submission form:")
print()
print(f"team:              {TEAM}")
print("case:              3")
print("round:             2  (Part II)")
print(f"decision:          {MIN_SEVERITY}")
print(f"predicted_profit:  {PREDICTED_PROFIT:.0f}")

## Submit (round 2)

**[Open the submission form](https://docs.google.com/forms/d/e/1FAIpQLSf-87eDsEHRVat_a4le91sC7bfZFel9fAHNmMtHVuGJzqaHOQ/viewform)** and enter the values printed above, exactly as shown.

- **Predicted profit:** a number such as `150000` (`$150,000` and `150K` also work, but not words).
- **Case:** `3`.  **Round:** `2` (Part II).
- **Submissions are final.** You can't change an answer after you submit, so check the printed values first. If your team submits more than once for the same round, only the **first** submission counts.
- Use the **same team name** in every round.

## Reflection (submit)

1. Your Part 1 analysis may have fit the data very well. Why was it still the wrong tool for this decision?
2. In one sentence: what was the difference between **predicting** the outcome and **deciding** what to do?
3. Name one other business decision where the same trap could appear.